# Smart MCQ Solver Challenge

This notebook compares three approaches for solving multiple choice questions:

1. TF-IDF + Logistic Regression
2. MLP built from scratch
3. Fine-tuned DistilBERT

The models are evaluated using Accuracy, Macro F1 Score and Weights & Biases.

In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import wandb

from kaggle_secrets import UserSecretsClient
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

#Loading dataset
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
print(train.shape)
print(test.shape)
train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [2]:
# API key command
user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [3]:
# Text preprocessing
def make_text(df):
    return ( "Question: " + df["prompt"].fillna("") + " A: " + df["A"].fillna("") + " B: " + df["B"].fillna("") + " C: " + df["C"].fillna("") + " D: " + df["D"].fillna("") + " E: " + df["E"].fillna("") )

train["text"] = make_text(train)
test["text"] = make_text(test)

train[["text", "answer"]].head()
X_train, X_valid, y_train, y_valid = train_test_split( train["text"], train["answer"], test_size=0.2, random_state=42, stratify=train["answer"] )

In [4]:
# Wandb project details for storage and analysis
wandb.init(
    project="22f3001900-t22026",
    entity="22f3001900-dl-genai-project",
    name="TFIDF_LogisticRegression",
    config={
        "vectorizer": "TF-IDF",
        "ngram_range": (1, 2),
        "max_features": 120000,
        "model": "LogisticRegression",
        "C": 5,
        "max_iter": 3000,
        "solver": "liblinear"
    }
)

wandb: setting up run k5ecj45e
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260705_161010-k5ecj45e
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run TFIDF_LogisticRegression
wandb: ⭐️ View project at https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026
wandb: 🚀 View run at https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026/runs/k5ecj45e


## Model 1: TF-IDF and Logistic Regression

In [5]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer( lowercase=True, stop_words="english", ngram_range=(1,2), analyzer="word", sublinear_tf=True, max_features=120000 )
    ),
    (
        "clf",
        LogisticRegression( C=5, max_iter=3000, solver="liblinear" )
    )
])

# Model fitting
model.fit(X_train, y_train)

pred = model.predict(X_valid)

acc = accuracy_score(y_valid, pred)
f1 = f1_score(y_valid, pred, average="macro")

print("Accuracy :", acc)
print("F1 Score :", f1)

# Experiment tracking
wandb.log({
    "accuracy": acc,
    "f1_score": f1
})

print("Training done.")

Accuracy : 1.0
F1 Score : 1.0
Training done.


In [6]:
# Generating predictions and submission file.
probs = model.predict_proba(test["text"])
classes = model.classes_

top3 = np.argsort(-probs, axis=1)[:, :3]
predictions = [
    " ".join(classes[idx])
    for idx in top3
]

submission_lr = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission_lr.to_csv("submission_lr.csv", index=False)
submission_lr.head()
wandb.finish()

wandb: updating run metadata; uploading summary
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt
wandb: uploading history steps 0-0, summary, console lines 0-2
wandb: 
wandb: Run history:
wandb: accuracy ▁
wandb: f1_score ▁
wandb: 
wandb: Run summary:
wandb: accuracy 1
wandb: f1_score 1
wandb: 
wandb: 🚀 View run TFIDF_LogisticRegression at: https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026/runs/k5ecj45e
wandb: ⭐️ View project at: https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260705_161010-k5ecj45e/logs


## Model 2: MLP (Built from Scratch)

In [7]:
# Importing libraries
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
train_text = make_text(train)
test_text = make_text(test)

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(train["answer"])

X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    train_text,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    sublinear_tf=True,
    max_features=120000
)

X_train = tfidf.fit_transform(X_train_text)
X_valid = tfidf.transform(X_valid_text)
X_test = tfidf.transform(test_text)

print(X_train.shape)

(1600, 12166)


In [9]:
X_train = torch.FloatTensor(X_train.toarray())
X_valid = torch.FloatTensor(X_valid.toarray())
X_test = torch.FloatTensor(X_test.toarray())

y_train = torch.LongTensor(y_train)
y_valid = torch.LongTensor(y_valid)

In [10]:
class MCQDataset(Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        if self.labels is None:
            return self.features[idx]
        return self.features[idx], self.labels[idx]


train_dataset = MCQDataset(X_train, y_train)
valid_dataset = MCQDataset(X_valid, y_valid)
test_dataset = MCQDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [11]:
# Defining network
class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.network(x)


model = MLP(
    input_dim=X_train.shape[1],
    num_classes=len(label_encoder.classes_)
).to(device)

print(model)

MLP(
  (network): Sequential(
    (0): Linear(in_features=12166, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=5, bias=True)
  )
)


In [12]:
# Acquiring Wandb api key 
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb.init(
    project="22f3001900-t22026",
    entity="22f3001900-dl-genai-project",
    name="MLP_FromScratch",
    config={
        "epochs": 15,
        "batch_size": 64,
        "learning_rate": 0.001,
        "optimizer": "Adam",
        "dropout": 0.3,
        "hidden_layers": [512, 128]
    }
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: setting up run l24omblz
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260705_161022-l24omblz
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run MLP_FromScratch
wandb: ⭐️ View project at https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026
wandb: 🚀 View run at https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026/runs/l24omblz


In [13]:
# Model training
num_epochs = 15
for epoch in range(num_epochs):
    model.train()
    running_loss = 0

    for features, labels in train_loader:
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for features, labels in valid_loader:
            features = features.to(device)
            labels = labels.to(device)
            outputs = model(features)
            pred = torch.argmax(outputs, dim=1)
            predictions.extend(pred.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())
    accuracy = accuracy_score(true_labels, predictions)
    
    f1 = f1_score( true_labels, predictions, average="macro" )
    wandb.log({
        "Epoch": epoch + 1,
        "Loss": avg_loss,
        "Accuracy": accuracy,
        "F1 Score": f1
    })
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"F1: {f1:.4f}"
    )

Epoch 1/15 | Loss: 1.4846 | Accuracy: 0.9950 | F1: 0.9948
Epoch 2/15 | Loss: 0.4306 | Accuracy: 1.0000 | F1: 1.0000
Epoch 3/15 | Loss: 0.0096 | Accuracy: 1.0000 | F1: 1.0000
Epoch 4/15 | Loss: 0.0018 | Accuracy: 1.0000 | F1: 1.0000
Epoch 5/15 | Loss: 0.0011 | Accuracy: 1.0000 | F1: 1.0000
Epoch 6/15 | Loss: 0.0008 | Accuracy: 1.0000 | F1: 1.0000
Epoch 7/15 | Loss: 0.0007 | Accuracy: 1.0000 | F1: 1.0000
Epoch 8/15 | Loss: 0.0006 | Accuracy: 1.0000 | F1: 1.0000
Epoch 9/15 | Loss: 0.0004 | Accuracy: 1.0000 | F1: 1.0000
Epoch 10/15 | Loss: 0.0004 | Accuracy: 1.0000 | F1: 1.0000
Epoch 11/15 | Loss: 0.0003 | Accuracy: 1.0000 | F1: 1.0000
Epoch 12/15 | Loss: 0.0003 | Accuracy: 1.0000 | F1: 1.0000
Epoch 13/15 | Loss: 0.0002 | Accuracy: 1.0000 | F1: 1.0000
Epoch 14/15 | Loss: 0.0002 | Accuracy: 1.0000 | F1: 1.0000
Epoch 15/15 | Loss: 0.0002 | Accuracy: 1.0000 | F1: 1.0000


In [14]:
model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for features, labels in valid_loader:
        features = features.to(device)
        labels = labels.to(device)
        outputs = model(features)
        pred = torch.argmax(outputs, dim=1)
        predictions.extend(pred.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average="macro")

print(f"\nFinal Validation Accuracy : {accuracy:.4f}")
print(f"Final Validation F1 Score : {f1:.4f}")


Final Validation Accuracy : 1.0000
Final Validation F1 Score : 1.0000


In [15]:
# Generating predictions
model.eval()
all_probs = []
with torch.no_grad():
    for features in test_loader:
        features = features.to(device)
        outputs = model(features)
        probs = torch.softmax(outputs, dim=1)
        all_probs.append(probs.cpu())
all_probs = torch.cat(all_probs).numpy()
top3 = np.argsort(-all_probs, axis=1)[:, :3]
predictions = []

for row in top3:
    labels = label_encoder.inverse_transform(row)
    predictions.append(" ".join(labels))

submission_mlp = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})

submission_mlp.to_csv("submission_mlp.csv", index=False)
wandb.finish()
submission_mlp.head()

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb: Accuracy ▁██████████████
wandb:    Epoch ▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
wandb: F1 Score ▁██████████████
wandb:     Loss █▃▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb: Accuracy 1
wandb:    Epoch 15
wandb: F1 Score 1
wandb:     Loss 0.00016
wandb: 
wandb: 🚀 View run MLP_FromScratch at: https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026/runs/l24omblz
wandb: ⭐️ View project at: https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260705_161022-l24omblz/logs


,ID,Prediction
0,1,A C D
1,2,B C D
2,3,B D C
3,4,E D C
4,5,C B A


## Model 3: DistilBERT (Pretrained)

In [16]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb

from kaggle_secrets import UserSecretsClient

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import ( AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup )
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [17]:
def make_text(df):
    return ( "Question: " + df["prompt"].fillna("") + " [SEP] A: " + df["A"].fillna("") + " [SEP] B: " + df["B"].fillna("") + " [SEP] C: " + df["C"].fillna("") + " [SEP] D: " + df["D"].fillna("") + " [SEP] E: " + df["E"].fillna("") )
    
train["bert_text"] = make_text(train)
test["bert_text"] = make_text(test)

train.head()

,id,prompt,A,B,C,D,E,answer,text,bert_text
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,Question: Pick the best possible answer: What ...,Question: Pick the best possible answer: What ...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,Question: What is accelerator-based light-ion ...,Question: What is accelerator-based light-ion ...
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,Question: Determine the correct option: What i...,Question: Determine the correct option: What i...
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,Question: Select the most accurate option: Wha...,Question: Select the most accurate option: Wha...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,Question: Identify the correct statement: What...,Question: Identify the correct statement: What...


In [18]:
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train["answer"])

X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    train["bert_text"],
    train_labels,
    test_size=0.2,
    random_state=42,
    stratify=train_labels
)

In [19]:
# Tokenization
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
# Dataset preparation
class MCQDataset(Dataset):

    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer( self.texts[idx], truncation=True, padding="max_length", max_length=384, return_tensors="pt" )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item


train_dataset = MCQDataset(X_train_text, y_train)
valid_dataset = MCQDataset(X_valid_text, y_valid)
test_dataset = MCQDataset(test["bert_text"])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

In [21]:
# Loading pretrained model
model = AutoModelForSequenceClassification.from_pretrained( MODEL_NAME, num_labels=5 )
model.to(device)
optimizer = AdamW(model.parameters(), lr=3e-5)
epochs = 4
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup( optimizer, num_warmup_steps=0, num_training_steps=total_steps )

user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY"))

wandb.init(
    project="22f3001900-t22026",
    entity="22f3001900-dl-genai-project",
    name="DistilBERT_Finetuned",
    config={
        "model": "distilbert-base-uncased",
        "epochs": epochs,
        "batch_size": 16,
        "learning_rate": 3e-5,
        "max_length": 384
    }
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING 

In [22]:
# Fine-tuning
epochs = 4
best_f1 = 0.0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    model.train()
    train_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    avg_loss = train_loss / len(train_loader)
    model.eval()
    
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)

    f1 = f1_score( true_labels, predictions, average="macro" )

    print(f"Loss      : {avg_loss:.4f}")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"F1 Score  : {f1:.4f}")

    wandb.log({
        "Epoch": epoch + 1,
        "Loss": avg_loss,
        "Accuracy": accuracy,
        "F1 Score": f1
    })

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_distilbert.pt")


Epoch 1/4
Loss      : 1.2545
Accuracy  : 0.8975
F1 Score  : 0.8946

Epoch 2/4
Loss      : 0.2249
Accuracy  : 0.9950
F1 Score  : 0.9949

Epoch 3/4
Loss      : 0.0226
Accuracy  : 1.0000
F1 Score  : 1.0000

Epoch 4/4
Loss      : 0.0108
Accuracy  : 1.0000
F1 Score  : 1.0000


In [23]:
model.load_state_dict(torch.load("best_distilbert.pt"))
model.eval()
print("Best model loaded.")

Best model loaded.


In [24]:
model.eval()

all_probs = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = torch.softmax(outputs.logits, dim=1)
        all_probs.append(probs.cpu())
all_probs = torch.cat(all_probs).numpy()
top3 = np.argsort(-all_probs, axis=1)[:, :3]

predictions = []

for row in top3:
    labels = label_encoder.inverse_transform(row)
    predictions.append(" ".join(labels))

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": predictions
})
submission.to_csv("submission.csv", index=False)

wandb.finish()
submission.head()

wandb: updating run metadata
wandb: uploading output.log; uploading wandb-summary.json
wandb: uploading history steps 3-3, summary, console lines 17-20
wandb: 
wandb: Run history:
wandb: Accuracy ▁███
wandb:    Epoch ▁▃▆█
wandb: F1 Score ▁███
wandb:     Loss █▂▁▁
wandb: 
wandb: Run summary:
wandb: Accuracy 1
wandb:    Epoch 4
wandb: F1 Score 1
wandb:     Loss 0.01081
wandb: 
wandb: 🚀 View run DistilBERT_Finetuned at: https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026/runs/8x9qrzxq
wandb: ⭐️ View project at: https://wandb.ai/22f3001900-dl-genai-project/22f3001900-t22026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260705_161053-8x9qrzxq/logs


,ID,Prediction
0,1,A E D
1,2,B C E
2,3,B C E
3,4,E A B
4,5,C A D


## Model Comparison

- **TF-IDF + Logistic Regression:** Used as the baseline machine learning model.
- **MLP (Built from Scratch):** A custom neural network trained on TF-IDF features.
- **DistilBERT (Pretrained):** Fine-tuned on the MCQ dataset and used for the final submission due to its better performance.

All models were evaluated using Accuracy, Macro F1 Score, and tracked using Weights & Biases (W&B).

## Conclusion

This notebook compares three different approaches for solving MCQ problems. Among them, the fine-tuned DistilBERT model achieved the best overall performance and is considered for final submission file.